In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 122 samples and 11995 columns.
First few rows of the data:


,Time to Recurrence,Progression,Days_to_Death or Last Contact,Dead_or_alive,Study_ID,Race,Male_1,stage,smoking_Pack_Years,Age_range,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,656,1,1141,1,20,White,1,IA,50.0,70-79,...,7.286328,4.373904,6.597366,7.722683,5.046826,6.580865,7.924363,7.167449,5.838908,7.102727
1,4661,0,4661,0,32,White,2,IA,5.0,60-69,...,7.065119,4.213735,6.424642,7.146611,5.949407,6.675288,7.809343,7.278092,5.831526,7.081395
2,2319,0,2319,0,50,White,1,1,7.5,70-79,...,9.408223,6.913720,9.183168,9.787556,8.098887,9.210414,10.177154,9.752128,8.031843,9.767943
3,4610,0,4610,0,62,White,1,1,45.0,60-69,...,7.675236,5.578374,7.355767,8.249087,6.519670,7.328719,8.565213,8.322208,6.614988,8.005188
4,3030,0,3030,0,113,White,2,1,35.0,70-79,...,7.829981,5.514712,7.691057,8.239129,6.501239,7.554302,8.892646,8.319649,6.650228,8.282383


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for Time to Recurrence or Progression: 0
Number of samples with complete data for both variables: 122
Number of censored samples (event_var == 0.0): 77
Number of events (event_var == 1.0): 45
Censored ratio (among complete): 77 / 122 = 0.631 (63.1%)
Number of samples with incomplete data for Days_to_Death or Last Contact or Dead_or_alive: 0
Number of samples with complete data for both variables: 122
Number of censored samples (event_var == 0.0): 89
Number of events (event_var == 1.0): 33
Censored ratio (among complete): 89 / 122 = 0.730 (73.0%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE229706/description.json
